In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import confusion_matrix

In [4]:
dataset_path = "/content/drive/MyDrive/Deep-Learning-Image-Classification/Dataset"


train_path = dataset_path + "/train"
test_path = dataset_path + "/test"
val_path = dataset_path + "/val"

In [5]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32

**Load Datasets**

In [6]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
train_path,
image_size=IMG_SIZE,
batch_size=BATCH_SIZE
)

Found 12632 files belonging to 6 classes.


In [7]:
val_dataset = tf.keras.utils.image_dataset_from_directory(
val_path,
image_size=IMG_SIZE,
batch_size=BATCH_SIZE
)

Found 1402 files belonging to 6 classes.


In [8]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
test_path,
image_size=IMG_SIZE,
batch_size=BATCH_SIZE,
shuffle=False
)

Found 3000 files belonging to 6 classes.


**Data Augmentation**

In [19]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.RandomZoom(0.1),

    tf.keras.layers.RandomTranslation(
        0.1,
        0.1
    )
])

**Load MobilenetV2**

In [20]:
base_model = MobileNetV2(

input_shape=(224,224,3),

include_top=False,

weights='imagenet'
)


**Freeze pretrained layers**

In [17]:
base_model.trainable = False

**Add Classification Head**

In [21]:
inputs = tf.keras.Input(
    shape=(224,224,3)
)

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.2)(x)

outputs = tf.keras.layers.Dense(
    6,
    activation='softmax'
)(x)

mobilenet_model = tf.keras.Model(
    inputs,
    outputs
)

**Compilation and summery view**

In [22]:
mobilenet_model.compile(

    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']
)

mobilenet_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 2,231,558 (8.51 MB)

 Non-trainable params: 34,112 (133.25 KB)

**Train Model**

In [ ]:
history = mobilenet_model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=10
)

**Evaluate model**

In [ ]:
test_loss, test_acc = mobilenet_model.evaluate(
test_dataset
)

print(test_acc)

**Confusion Matrix**

In [ ]:
cm = confusion_matrix(
y_true,
y_pred
)

**Save Model**

In [ ]:
mobilenet_model.save(
'/content/drive/MyDrive/Deep-Learning-Image-Classification/Models/mobilenetv2_feature_extraction.keras'
)

**Save Training History**

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history.history)

history_df.to_csv(
'/content/drive/MyDrive/Deep-Learning-Image-Classification/Results/mobilenetv2_history.csv',
index=False
)